# Stellar Classification - Hyperparameter Tuning

This notebook executes Optuna hyperparameter search on a 20% stratified sample of the training dataset using 3-Fold Stratified CV.

In [ ]:
import os
import numpy as np
import pandas as pd
import optuna
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
if os.path.exists('/kaggle/input/competitions/playground-series-s6e6'):
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
    OUTPUT_DIR = '.'
else:
    DATA_DIR = '../data'
    OUTPUT_DIR = '..'

TARGET_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
SPECTRAL_MAP = {'M': 0, 'A/F': 1, 'G/K': 2, 'O/B': 3}
GALAXY_POP_MAP = {'Red_Sequence': 0, 'Blue_Cloud': 1}

train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

def feature_engineering(df):
    df = df.copy()
    df['spectral_type_code'] = df['spectral_type'].map(SPECTRAL_MAP).fillna(-1).astype(int)
    df['galaxy_pop_code'] = df['galaxy_population'].map(GALAXY_POP_MAP).fillna(-1).astype(int)
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['r_z'] = df['r'] - df['z']
    df['u_i'] = df['u'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['u_z'] = df['u'] - df['z']
    df['stellar_color_bin'] = pd.cut(df['g_r'], bins=[-np.inf, 0.0, 0.4, 0.8, 1.2, np.inf], labels=[0, 1, 2, 3, 4]).fillna(2).astype(int)
    
    alpha_rad = np.radians(df['alpha'])
    delta_rad = np.radians(df['delta'])
    df['coord_x'] = np.cos(delta_rad) * np.cos(alpha_rad)
    df['coord_y'] = np.cos(delta_rad) * np.sin(alpha_rad)
    df['coord_z'] = np.sin(delta_rad)
    df['coord_x_redshift'] = df['redshift'] * df['coord_x']
    df['coord_y_redshift'] = df['redshift'] * df['coord_y']
    df['coord_z_redshift'] = df['redshift'] * df['coord_z']
    
    # Galactic Coordinates
    x_gal = -0.05487554 * df['coord_x'] - 0.87343710 * df['coord_y'] - 0.48383499 * df['coord_z']
    y_gal = 0.49410945 * df['coord_x'] - 0.44482959 * df['coord_y'] + 0.74698225 * df['coord_z']
    z_gal = -0.86766614 * df['coord_x'] - 0.19807639 * df['coord_y'] + 0.45598380 * df['coord_z']
    df['gal_x'] = x_gal
    df['gal_y'] = y_gal
    df['gal_z'] = z_gal
    df['gal_l'] = np.degrees(np.arctan2(y_gal, x_gal)) % 360
    df['gal_b'] = np.degrees(np.arcsin(z_gal))
    
    # Redshift interactions
    df['u_g_redshift'] = df['u_g'] * df['redshift']
    df['g_r_redshift'] = df['g_r'] * df['redshift']
    df['r_i_redshift'] = df['r_i'] * df['redshift']
    df['i_z_redshift'] = df['i_z'] * df['redshift']
    df['u_r_redshift'] = df['u_r'] * df['redshift']
    df['g_i_redshift'] = df['g_i'] * df['redshift']
    df['r_z_redshift'] = df['r_z'] * df['redshift']
    df['u_i_redshift'] = df['u_i'] * df['redshift']
    df['g_z_redshift'] = df['g_z'] * df['redshift']
    df['u_z_redshift'] = df['u_z'] * df['redshift']
    
    # Curvature
    df['ug_gr'] = df['u_g'] - df['g_r']
    df['ug_ri'] = df['u_g'] - df['r_i']
    df['ug_iz'] = df['u_g'] - df['i_z']
    df['gr_ri'] = df['g_r'] - df['r_i']
    df['gr_iz'] = df['g_r'] - df['i_z']
    df['ri_iz'] = df['r_i'] - df['i_z']
    
    cols_to_drop = ['id', 'spectral_type', 'galaxy_population']
    return df.drop(columns=[col for col in cols_to_drop if col in df.columns])

X_full = feature_engineering(train_raw)
y_full = X_full.pop('class').map(TARGET_MAP).values

# Extract 20% Stratified Subset
SEED = 42
skf_sub = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
_, sub_idx = next(skf_sub.split(X_full, y_full))
X = X_full.iloc[sub_idx].reset_index(drop=True)
y = y_full[sub_idx]
print(f"Tuning Subset Shape: {X.shape}")